# 04 · Đánh giá, phân tích lỗi và giới hạn
Chỉ đọc kết quả, không huấn luyện. Nhãn 1 = nguy cơ (khác notebook legacy: 1 = Đỗ). Bài giảng: trang 21–22, 58–67, 107–108.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
assert (ROOT / 'src').exists(), 'Open notebook from project or notebooks directory'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.data_engine import PROFILE_FILE, COURSE_FILE, MODEL_DIR, FEATURES
profiles = pd.read_csv(PROFILE_FILE)
records = pd.read_csv(COURSE_FILE)


In [ ]:
import json
meta = json.loads((MODEL_DIR / 'metadata.json').read_text(encoding='utf-8'))
metrics = pd.read_csv(MODEL_DIR / 'metrics.csv')
predictions = pd.read_csv(MODEL_DIR / 'test_predictions.csv')
display(metrics[['Model', 'Accuracy', 'Recall_NguyCo', 'Precision_NguyCo', 'F1_NguyCo', 'F2_NguyCo', 'Average_Precision', 'ROC_AUC', 'Threshold', 'Selected']])
print('Selected from train CV:', meta['selected_model'])

In [ ]:
from src.experiments import scores
p = predictions.loc[predictions.Model.eq(meta['selected_model'])]
display(pd.Series(scores(p.NguyCo, p.Probability, meta['threshold'])))
display(p.groupby(['HocPhan', 'Error']).size().unstack(fill_value=0))
display(p.loc[p.Error.eq('FN')].head(10))

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
PrecisionRecallDisplay.from_predictions(p.NguyCo, p.Probability, ax=axes[0])
RocCurveDisplay.from_predictions(p.NguyCo, p.Probability, ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
learning = pd.read_csv(MODEL_DIR / 'learning_curve.csv')
learning.groupby('Fraction')[['Train_AP', 'Validation_AP']].mean().plot(marker='o', title='Learning diagnostic: ' + meta['selected_model'])
plt.show()
display(learning.groupby('Fraction')[['Train_AP', 'Validation_AP']].agg(['mean', 'std']))

## Biện luận bắt buộc
1. So sánh với Dummy: Accuracy cao nhưng Recall có thể bằng 0.
2. Phân biệt AP (Average Precision) với diện tích PR theo hình thang.
3. Đường cong học là chẩn đoán với cấu hình đã chọn; không phải kết quả độc lập.
4. Nếu chỉnh mô hình từ phân tích lỗi test, cần tập đánh giá mới.
5. Không kết luận mô hình thắng tuyệt đối khi chênh lệch CV nhỏ.
6. Dữ liệu mô phỏng, xác suất chưa kiểm định calibration, What-if không có nghĩa nhân quả.
7. SVC probability=True còn dùng được ở sklearn 1.9 nhưng đã bị đánh dấu deprecated; cần chuyển sang calibration theo nhóm khi nâng phiên bản.
8. Không áp dụng quyết định học vụ tự động từ bản demo.